# YOLO26-Depth — Pothole-440 Fine-Tuning

This notebook fine-tunes **Ultralytics YOLO26n-Depth** on the prepared Pothole-440 dataset.

It uses:
- RGB images from `Pothole440_Grouped`
- Globally normalized scanner-intensity targets from `Pothole440_Targets_Global`
- Corrected scanner validity masks from `Pothole440_Targets_Corrected`

## Important target interpretation

The Pothole-440 target is a **globally normalized scanner-grayscale proxy in [0, 1]**, not calibrated metric depth in meters. The scanner's exact depth encoding/scale is not established. The YOLO26 depth task is designed around positive depth targets and treats values `<= 0` as invalid. This notebook therefore exports the normalized target as positive float32 `.npy` values and writes invalid pixels as zero.

Consequences:
- The network is learning the prepared relative scanner-intensity target, not physical distance in meters.
- Ultralytics' native depth metrics are designed for metric depth and must not be reported as physical-unit accuracy for this experiment.
- Final comparison should use a common external evaluation protocol against the same normalized target and corrected validity masks for all four models.
- The test split is included in the dataset layout but is not used for training or checkpoint selection.

This notebook starts from **pretrained YOLO26n-Depth weights** and fine-tunes them. It does not train the architecture from random initialization.


## 1. Environment and installation

In [1]:
# Kaggle: enable GPU in Notebook Settings before running.
!pip -q install -U ultralytics

import os
import gc
import json
import shutil
import random
import numpy as np
import pandas as pd
import yaml
import torch
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__)
print("Ultralytics:", __import__("ultralytics").__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
PyTorch: 2.11.0+cu130
Ultralytics: 8.4.216
CUDA available: True
GPU: Tesla T4


## 2. Configure preprocessing output paths

Attach the saved preprocessing notebook output to this notebook through Kaggle **Add Input → Notebook Output Files**. That output should contain `Pothole440_Grouped/`, `Pothole440_Targets_Global/`, and `Pothole440_Targets_Corrected/`.

In [2]:
# Change this to the mounted preprocessing notebook output path in Kaggle.
# If all preparation folders are in the same notebook output root, leave the subfolder names below.
PREP_DIR = Path("/kaggle/input/datasets/siddharthk7704/pothole-dataset")

GROUPED_ROOT = PREP_DIR / "Pothole440_Grouped"
GLOBAL_ROOT = PREP_DIR / "Pothole440_Targets_Global"
CORRECTED_ROOT = PREP_DIR / "Pothole440_Targets_Corrected"
MANIFEST_PATH = CORRECTED_ROOT / "training_manifest_corrected.csv"

OUTPUT_DIR = Path("/kaggle/working/YOLO26_Depth_Pothole440")
DATASET_DIR = OUTPUT_DIR / "dataset"
YAML_PATH = OUTPUT_DIR / "pothole440_depth.yaml"
RUNS_DIR = OUTPUT_DIR / "runs"

for p in [GROUPED_ROOT, GLOBAL_ROOT, CORRECTED_ROOT, MANIFEST_PATH]:
    print(f"{p}: {p.exists()}")

assert GROUPED_ROOT.is_dir(), f"Missing grouped dataset: {GROUPED_ROOT}"
assert CORRECTED_ROOT.is_dir(), f"Missing corrected targets: {CORRECTED_ROOT}"
assert MANIFEST_PATH.is_file(), f"Missing corrected manifest: {MANIFEST_PATH}"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


/kaggle/input/datasets/siddharthk7704/pothole-dataset/Pothole440_Grouped: True
/kaggle/input/datasets/siddharthk7704/pothole-dataset/Pothole440_Targets_Global: True
/kaggle/input/datasets/siddharthk7704/pothole-dataset/Pothole440_Targets_Corrected: True
/kaggle/input/datasets/siddharthk7704/pothole-dataset/Pothole440_Targets_Corrected/training_manifest_corrected.csv: True


## 3. Load manifest and resolve prepared file paths

The manifest may retain `/kaggle/working/...` paths from the preprocessing notebook. `resolve_path()` remaps those paths to the attached Kaggle input root and checks the actual files.

In [3]:
df = pd.read_csv(MANIFEST_PATH)
print("Manifest shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head(3))

def choose_col(frame, candidates, label):
    for c in candidates:
        if c in frame.columns:
            return c
    raise KeyError(f"Could not identify {label}. Tried {candidates}; columns={frame.columns.tolist()}")

SPLIT_COL = choose_col(df, ["split"], "split")
NAME_COL = choose_col(df, ["filename", "file_name"], "filename")
RGB_COL = choose_col(df, ["rgb_path", "image_path", "image"], "RGB image path")
TARGET_COL = choose_col(df, ["target_path", "depth_target_path"], "normalized target path")
VALID_COL = choose_col(df, ["corrected_valid_path", "valid_path", "valid_mask_path"], "corrected validity-mask path")

def resolve_path(value):
    if pd.isna(value):
        return None
    raw = str(value)
    p = Path(raw)
    if p.is_file():
        return p

    # Paths created in the preprocessing notebook commonly begin with /kaggle/working/.
    marker = "/kaggle/working/"
    if raw.startswith(marker):
        candidate = PREP_DIR / raw[len(marker):]
        if candidate.is_file():
            return candidate

    # Try paths relative to the attached preparation root.
    candidate = PREP_DIR / raw.lstrip("/")
    if candidate.is_file():
        return candidate

    # Last resort: locate by basename under the expected source roots.
    basename = p.name
    for root in [GROUPED_ROOT, GLOBAL_ROOT, CORRECTED_ROOT]:
        for found in root.rglob(basename):
            if found.is_file():
                return found
    return p

df["rgb_resolved"] = df[RGB_COL].apply(resolve_path)
df["target_resolved"] = df[TARGET_COL].apply(resolve_path)
df["valid_resolved"] = df[VALID_COL].apply(resolve_path)

missing = {
    "rgb": int((~df["rgb_resolved"].apply(lambda p: p is not None and p.is_file())).sum()),
    "target": int((~df["target_resolved"].apply(lambda p: p is not None and p.is_file())).sum()),
    "valid": int((~df["valid_resolved"].apply(lambda p: p is not None and p.is_file())).sum()),
}
print("Missing paths:", missing)
print("Split counts:")
display(df[SPLIT_COL].value_counts().rename_axis("split").to_frame("images"))
assert sum(missing.values()) == 0, "Some manifest paths could not be resolved. Check PREP_DIR and the manifest."


Manifest shape: (435, 11)
Columns: ['split', 'filename', 'rgb_path', 'mask_path', 'target_path', 'valid_path', 'height', 'width', 'valid_pixels', 'valid_fraction', 'corrected_valid_path']


,split,filename,rgb_path,mask_path,target_path,valid_path,height,width,valid_pixels,valid_fraction,corrected_valid_path
0,train,test__2022_05_22_pothole_10_horizantal_aug1.png,/kaggle/working/Pothole440_Grouped/train/rgb/t...,/kaggle/working/Pothole440_Grouped/train/mask/...,/kaggle/working/Pothole440_Targets_Global/trai...,/kaggle/working/Pothole440_Targets_Global/trai...,800,1200,113419,0.118145,/kaggle/working/Pothole440_Targets_Global/trai...
1,train,test__2022_05_22_pothole_11_horizantal_vertica...,/kaggle/working/Pothole440_Grouped/train/rgb/t...,/kaggle/working/Pothole440_Grouped/train/mask/...,/kaggle/working/Pothole440_Targets_Global/trai...,/kaggle/working/Pothole440_Targets_Global/trai...,800,1200,115216,0.120017,/kaggle/working/Pothole440_Targets_Global/trai...
2,train,test__2022_05_22_pothole_11_horizantal_vertica...,/kaggle/working/Pothole440_Grouped/train/rgb/t...,/kaggle/working/Pothole440_Grouped/train/mask/...,/kaggle/working/Pothole440_Targets_Global/trai...,/kaggle/working/Pothole440_Targets_Global/trai...,800,1200,115376,0.120183,/kaggle/working/Pothole440_Targets_Global/trai...


Missing paths: {'rgb': 0, 'target': 0, 'valid': 0}
Split counts:


,images
split,
train,305
val,66
test,64


## 4. Build Ultralytics depth dataset layout

Ultralytics expects paired files with matching stems:

```text
YOLO26_Depth_Pothole440/dataset/
├── images/train, images/val, images/test
└── depth/train, depth/val, depth/test
```

For each RGB image, the paired target is a float32 `.npy` file with the same stem. The corrected validity mask is applied during conversion: invalid target pixels are set to `0`, which Ultralytics excludes from depth loss and depth metrics. The target values remain the normalized scanner proxy; they are not converted to meters.

In [4]:
# Rebuild export folders to avoid stale pairs from earlier runs.
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)

for split in ["train", "val", "test"]:
    (DATASET_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / "depth" / split).mkdir(parents=True, exist_ok=True)

def canonical_split(value):
    s = str(value).lower()
    if s in ["validation", "valid"]:
        return "val"
    if s not in ["train", "val", "test"]:
        raise ValueError(f"Unexpected split label: {value}")
    return s

conversion_rows = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Export YOLO depth pairs"):
    split = canonical_split(row[SPLIT_COL])
    rgb_path = Path(row["rgb_resolved"])
    target_path = Path(row["target_resolved"])
    valid_path = Path(row["valid_resolved"])

    filename = str(row[NAME_COL])
    stem = Path(filename).stem
    image_ext = rgb_path.suffix.lower()
    if image_ext not in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
        image_ext = ".png"

    out_image = DATASET_DIR / "images" / split / f"{stem}{image_ext}"
    out_depth = DATASET_DIR / "depth" / split / f"{stem}.npy"

    # Copy RGB without altering pixel values.
    shutil.copy2(rgb_path, out_image)

    # Load normalized target
    if str(target_path).lower().endswith(".npy"):
        target = np.load(target_path).astype(np.float32)
    else:
        target = np.asarray(
            Image.open(target_path).convert("F"),
            dtype=np.float32
        )
    
    # Load corrected validity mask
    if str(valid_path).lower().endswith(".npy"):
        valid = np.load(valid_path)
    else:
        valid = np.asarray(
            Image.open(valid_path).convert("L")
        )
    
    valid = valid > 0

    if target.shape != valid.shape:
        raise ValueError(f"Target/mask shape mismatch for {filename}: {target.shape} vs {valid.shape}")

    # Ensure finite, nonnegative target; set all invalid pixels to zero.
    target = np.nan_to_num(target, nan=0.0, posinf=0.0, neginf=0.0)
    target = np.clip(target, 0.0, 1.0)
    target[~valid] = 0.0
    target = target.astype(np.float32)

    if np.count_nonzero(target > 0) < 10:
        raise ValueError(f"Too few valid positive target pixels for {filename}")

    np.save(out_depth, target)

    conversion_rows.append({
        "split": split,
        "filename": filename,
        "image_path": str(out_image),
        "depth_path": str(out_depth),
        "height": int(target.shape[0]),
        "width": int(target.shape[1]),
        "valid_pixels": int(np.count_nonzero(target > 0)),
        "valid_fraction": float(np.mean(target > 0)),
        "target_min_positive": float(target[target > 0].min()),
        "target_max": float(target.max())
    })

conversion_df = pd.DataFrame(conversion_rows)
conversion_df.to_csv(OUTPUT_DIR / "yolo26_dataset_pairs.csv", index=False)
display(conversion_df.groupby("split").agg(
    images=("filename", "count"),
    mean_valid_fraction=("valid_fraction", "mean"),
    min_valid_fraction=("valid_fraction", "min"),
    max_valid_fraction=("valid_fraction", "max")
))
print("Exported pairs:", len(conversion_df))


Export YOLO depth pairs:   0%|          | 0/435 [00:00<?, ?it/s]

,images,mean_valid_fraction,min_valid_fraction,max_valid_fraction
split,,,,
test,64,0.281439,0.097445,0.950761
train,305,0.225537,0.052270,0.941139
val,66,0.276777,0.089929,0.858885


Exported pairs: 435


## 5. Verify image-depth pairing and target ranges

In [5]:
pair_errors = []
for _, r in conversion_df.iterrows():
    ip = Path(r["image_path"])
    dp = Path(r["depth_path"])
    if not ip.is_file() or not dp.is_file():
        pair_errors.append((str(ip), str(dp), "missing"))
        continue
    arr = np.load(dp)
    if arr.dtype != np.float32 or arr.ndim != 2:
        pair_errors.append((str(dp), str(arr.dtype), str(arr.shape)))
    if not np.isfinite(arr).all():
        pair_errors.append((str(dp), "non-finite values"))

print("Pair errors:", len(pair_errors))
if pair_errors[:10]:
    print(pair_errors[:10])

assert len(pair_errors) == 0, "Dataset pairing validation failed."

print("Target statistics:")
all_positive = []
for p in conversion_df["depth_path"]:
    a = np.load(p)
    v = a[a > 0]
    all_positive.append(v)
vals = np.concatenate(all_positive)
print("Positive target count:", len(vals))
print("Min / median / max:", float(vals.min()), float(np.median(vals)), float(vals.max()))


Pair errors: 0
Target statistics:
Positive target count: 100865442
Min / median / max: 0.006369426846504211 0.6942675113677979 1.0


## 6. Create dataset YAML

The test split is declared for later use but will not be passed to training or checkpoint selection.

In [6]:
dataset_yaml = {
    "path": str(DATASET_DIR),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 1,
    "names": {0: "depth"},
    "channels": 3
}

with open(YAML_PATH, "w") as f:
    yaml.safe_dump(dataset_yaml, f, sort_keys=False)

print(YAML_PATH.read_text())


path: /kaggle/working/YOLO26_Depth_Pothole440/dataset
train: images/train
val: images/val
test: images/test
nc: 1
names:
  0: depth
channels: 3



## 7. Load pretrained YOLO26n-Depth

The nano variant is a sensible starting point for a Tesla T4 with limited memory. Ultralytics recommends fine-tuning pretrained depth weights with AdamW and a low learning rate rather than the default from-scratch optimizer settings.

In [7]:
MODEL_WEIGHTS = "yolo26n-depth.pt"
model = YOLO(MODEL_WEIGHTS)

print("Loaded:", MODEL_WEIGHTS)
print("Model task:", model.task)
assert model.task == "depth", f"Expected depth task, got {model.task}"


Loaded: yolo26n-depth.pt
Model task: depth


## 8. Fine-tune YOLO26n-Depth

The official custom fine-tuning guidance recommends AdamW and a low learning rate. This run uses 20 epochs as an initial budget, with best-checkpoint selection by validation performance. If GPU memory is insufficient, reduce `BATCH_SIZE` to 2.

In [8]:
EPOCHS = 20
IMGSZ = 640
BATCH_SIZE = 4
WORKERS = 2

train_results = model.train(
    data=str(YAML_PATH),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH_SIZE,
    workers=WORKERS,
    device=DEVICE,
    optimizer="AdamW",
    lr0=1e-4,
    lrf=0.01,
    warmup_bias_lr=1e-4,
    weight_decay=1e-4,
    patience=6,
    cos_lr=True,
    amp=True,
    project=str(RUNS_DIR),
    name="yolo26n_depth_pothole440",
    exist_ok=True,
    plots=True,
    save=True,
    verbose=True
)

print("Training finished.")
print("Run directory:", train_results.save_dir)


Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/YOLO26_Depth_Pothole440/pothole440_depth.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-depth.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26n_depth_pothole

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/20     0.992G     0.2523    0.01597          1        640: 100% ━━━━━━━━━━━━ 77/77 10.9it/s 7.1s
                 Class     Images     delta1    abs_rel       rmse      silog: 100% ━━━━━━━━━━━━ 17/17 17.0it/s 1.0s
             depth_val         66     0.8641     0.1619     0.1033      17.65

      Epoch    GPU_mem  dlog_loss dgrad_loss  Instances       Size
       3/20     0.992G     0.2335    0.01552          1        640: 100% ━━━━━━━━━━━━ 77/77 11.6it/s 6.6s
                 Class     Images     delta1    abs_rel       rmse      silog: 100% ━━━━━━━━━━━━ 17/17 14.5it/s 1.2s
             depth_val         66     0.8912     0.1563    0.09744      17.01

      Epoch    GPU_mem  dlog_loss dgrad_loss  Instances       Size
       4/20     0.992G     0.2211    0.01583          1        640: 100% ━━━━━━━━━━━━ 77/77 11.0it/s 7.0s
                 Class     Images     delta1    abs_rel       rmse      silog: 100% ━━━━━━━━━━━━ 17/17 16.3it/s 1.0s
             depth_val         66      

## 9. Locate and load the best checkpoint

In [9]:
run_dir = Path(train_results.save_dir)
best_ckpt = run_dir / "weights" / "best.pt"
last_ckpt = run_dir / "weights" / "last.pt"

print("Best checkpoint exists:", best_ckpt.is_file(), best_ckpt)
print("Last checkpoint exists:", last_ckpt.is_file(), last_ckpt)
assert best_ckpt.is_file(), "best.pt was not produced."

best_model = YOLO(str(best_ckpt))
print("Loaded best model task:", best_model.task)


Best checkpoint exists: True /kaggle/working/YOLO26_Depth_Pothole440/runs/yolo26n_depth_pothole440/weights/best.pt
Last checkpoint exists: True /kaggle/working/YOLO26_Depth_Pothole440/runs/yolo26n_depth_pothole440/weights/last.pt
Loaded best model task: depth


## 10. Validation using Ultralytics depth metrics

These built-in metrics are useful as training diagnostics, but their metric-depth interpretation does not apply to this scanner-intensity proxy. Use the shared external normalized-target evaluation later for the four-model comparison.

In [10]:
val_metrics = best_model.val(
    data=str(YAML_PATH),
    split="val",
    imgsz=IMGSZ,
    batch=BATCH_SIZE,
    device=DEVICE,
    plots=True,
    project=str(RUNS_DIR),
    name="yolo26n_depth_validation",
    exist_ok=True
)

print("Native YOLO26 validation metrics (proxy-target diagnostics only):")
for attr in ["delta1", "delta2", "delta3", "abs_rel", "rmse", "silog"]:
    value = getattr(val_metrics, attr, None)
    print(f"{attr}: {value}")


Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n-depth-log summary (fused): 106 layers, 5,165,705 parameters, 0 gradients, 32.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5541.3±1014.5 MB/s, size: 1471.9 KB)
val: Scanning /kaggle/working/YOLO26_Depth_Pothole440/dataset/depth/val.cache... 66 images, 0 missing depth, 0 corrupt: 100% ━━━━━━━━━━━━ 66/66 15.4Mit/s 0.0s
                 Class     Images     delta1    abs_rel       rmse      silog: 100% ━━━━━━━━━━━━ 17/17 8.0it/s 2.1s
             depth_val         66      0.908     0.1497    0.09069      16.25
Speed: 3.1ms preprocess, 18.8ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /kaggle/working/YOLO26_Depth_Pothole440/runs/yolo26n_depth_validation
Native YOLO26 validation metrics (proxy-target diagnostics only):
delta1: 0.9079933934139482
delta2: 0.976681902553096
delta3: 0.9864959003347339
abs_rel: 0.14971594134289207
rmse: 0.09069410753859715
silog: 16.2

## 11. Generate validation predictions and save raw float maps

In [11]:
VAL_PRED_DIR = OUTPUT_DIR / "validation_predictions"
VAL_PRED_DIR.mkdir(parents=True, exist_ok=True)

val_rows = conversion_df[conversion_df["split"] == "val"].reset_index(drop=True)
val_pred_records = []

for _, row in tqdm(val_rows.iterrows(), total=len(val_rows), desc="YOLO26 validation inference"):
    image_path = row["image_path"]
    result = best_model.predict(
        source=image_path,
        imgsz=IMGSZ,
        device=DEVICE,
        verbose=False
    )[0]

    pred = result.depth.data.detach().float().cpu().numpy()
    stem = Path(image_path).stem

    np.save(VAL_PRED_DIR / f"{stem}.npy", pred.astype(np.float32))

    # Store a per-image normalized preview only for visual inspection.
    lo, hi = np.percentile(pred[np.isfinite(pred)], [1, 99])
    preview = np.clip((pred - lo) / max(hi - lo, 1e-8), 0, 1)
    Image.fromarray((preview * 255).astype(np.uint8)).save(
        VAL_PRED_DIR / f"{stem}_preview.png"
    )

    val_pred_records.append({
        "filename": row["filename"],
        "prediction_path": str(VAL_PRED_DIR / f"{stem}.npy"),
        "prediction_min": float(np.nanmin(pred)),
        "prediction_max": float(np.nanmax(pred)),
        "prediction_mean": float(np.nanmean(pred))
    })

pd.DataFrame(val_pred_records).to_csv(
    OUTPUT_DIR / "validation_prediction_manifest.csv",
    index=False
)
print("Saved validation predictions:", len(val_pred_records))


YOLO26 validation inference:   0%|          | 0/66 [00:00<?, ?it/s]

Saved validation predictions: 66


## 12. Save experiment metadata

In [12]:
summary = {
    "model": "Ultralytics YOLO26n-Depth",
    "weights_initialization": MODEL_WEIGHTS,
    "task": "depth",
    "epochs_requested": EPOCHS,
    "image_size": IMGSZ,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "initial_learning_rate": 1e-4,
    "train_images": int((conversion_df["split"] == "train").sum()),
    "validation_images": int((conversion_df["split"] == "val").sum()),
    "test_images_available_but_not_used": int((conversion_df["split"] == "test").sum()),
    "target": "globally normalized scanner-grayscale proxy in [0,1], not metric depth",
    "invalid_pixel_encoding": "0; excluded by Ultralytics depth loader",
    "best_checkpoint": str(best_ckpt),
    "validation_predictions": str(VAL_PRED_DIR),
    "test_evaluated": False
}

with open(OUTPUT_DIR / "experiment_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))


{
  "model": "Ultralytics YOLO26n-Depth",
  "weights_initialization": "yolo26n-depth.pt",
  "task": "depth",
  "epochs_requested": 20,
  "image_size": 640,
  "batch_size": 4,
  "optimizer": "AdamW",
  "initial_learning_rate": 0.0001,
  "train_images": 305,
  "validation_images": 66,
  "test_images_available_but_not_used": 64,
  "target": "globally normalized scanner-grayscale proxy in [0,1], not metric depth",
  "invalid_pixel_encoding": "0; excluded by Ultralytics depth loader",
  "best_checkpoint": "/kaggle/working/YOLO26_Depth_Pothole440/runs/yolo26n_depth_pothole440/weights/best.pt",
  "validation_predictions": "/kaggle/working/YOLO26_Depth_Pothole440/validation_predictions",
  "test_evaluated": false
}


## 13. Test-set evaluation — run only after all four models are finalized

Do not run this cell until MiDaS, DA V1, DA V2 and YOLO26 checkpoints have all been selected using validation data. Then evaluate each model once on the same 64-image test set using the shared corrected validity masks and the common normalized-target metrics.

The YOLO26 model's predicted depth values may have a different scale from the normalized target. For the fair comparison, use the same explicit evaluation/calibration protocol for all models, and report any scale alignment separately. Do not label the scanner target or predictions as meters.
